# Dr.Copper v2 - Phase 4: XGB

In [1]:
import warnings

import pandas as pd
import plotly.io as pio
from IPython.display import HTML, display

import dr_copper_v2 as dc
from dr_copper_v2.data_loader import ROOT

pio.renderers.default = "plotly_mimetype+png"
pio.renderers["png"].scale = 2

warnings.filterwarnings("ignore")

## 1 - Load development inputs

In [2]:
data_dir = ROOT / "data" / "processed"
raw = pd.read_parquet(data_dir / "raw_development.parquet")
features = pd.read_parquet(data_dir / "features_development.parquet")
garch = pd.read_parquet(data_dir / "garch" / "gaussian_forecasts_development.parquet")
splits = pd.read_parquet(data_dir / "pca" / "n_components_12" / "splits.parquet")

## 2 - Five-row direction target

Five sessions means five rows in the prepared raw table. Build the target before matching feature rows. Strictly positive returns are up; zero returns are non-up.

In [3]:
targets = dc.direction_target(raw["copper"])

garch_columns = ["volatility_1d", "volatility_5d"]
input_rows = features.index[features.notna().all(axis=1)]
garch_rows = garch.index[garch[garch_columns].notna().all(axis=1)]
input_rows = input_rows.intersection(garch_rows).sort_values()
labels = targets.loc[input_rows]

labels.head()

,label_end,forward_return,direction
date,,,
2013-01-02,2013-01-09,-0.015763,0
2013-01-03,2013-01-10,-0.012727,0
2013-01-04,2013-01-11,-0.010093,0
2013-01-07,2013-01-14,-0.003906,0
2013-01-08,2013-01-15,-0.003906,0


## 3 - Monthly training eligibility

At each saved cutoff, use only earlier input rows whose outcome has matured (`label_end <= train_end`). This applies de Prado's overlap-purging principle (2018, §12.2, pp. 161–162). The already fitted outer PCA/KMeans representations use feature history from 2012; supervised eligibility is a separate filter.

In [4]:
folds = {}
for split in splits.itertuples():
    train = labels.loc[split.train_start : split.train_end]
    train = train.loc[
        train["direction"].notna() & train["label_end"].le(split.train_end)
    ]
    forecast = labels.loc[split.test_start : split.test_end]
    folds[split.test_start.strftime("%Y-%m")] = {
        "train_labels": train,
        "forecast_labels": forecast,
    }

## 4 - Assemble the four feature groups

Regime IDs are fold-local categories, encoded into two indicator columns.

In [5]:
pca_dir = data_dir / "pca" / "n_components_12"
for month, fold in folds.items():
    fold["inputs"] = {}
    for side, artifact in [("train", "train"), ("forecast", "test")]:
        index = fold[f"{side}_labels"].index
        scores = pd.read_parquet(pca_dir / month / f"{artifact}_scores.parquet")
        regimes = pd.read_parquet(
            pca_dir / "kmeans_2" / month / f"{artifact}_labels.parquet"
        )
        fold["inputs"][side] = dc.direction_feature_groups(
            features.loc[index], garch, scores, regimes
        )

### Structure of `folds`

Each month contains label tables and four feature groups for training and forecasting. Every leaf is a date-indexed DataFrame.

```text
folds
└── "2017-01"  (one of 72 months)
    ├── train_labels       — matured training outcomes
    ├── forecast_labels    — forecast-month outcomes
    └── inputs
        ├── train
        │   ├── market_garch
        │   ├── market_macro_garch
        │   ├── market_macro_garch_pc_regimes
        │   └── garch_pc_regimes
        └── forecast
            ├── market_garch
            ├── market_macro_garch
            ├── market_macro_garch_pc_regimes
            └── garch_pc_regimes
```

Training feature rows match `train_labels`; forecast feature rows match `forecast_labels`. For example:

```python
fold = folds["2017-01"]
X_train = fold["inputs"]["train"]["market_garch"]
y_train = fold["train_labels"]["direction"]
X_forecast = fold["inputs"]["forecast"]["market_garch"]
```


## 5 - Joint Optuna search

One trial chooses a feature group and XGBoost hyperparameters, then refits monthly across all 72 Data2 folds. Maximize pooled direction accuracy on the same dates; output strictly above 0.5 means up.

In [6]:
import optuna

feature_groups = [
    "market_garch",
    "market_macro_garch",
    "market_macro_garch_pc_regimes",
    "garch_pc_regimes",
]
xgb_fixed = {
    "objective": "binary:logistic",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": 1,
}


def objective(trial):
    group = trial.suggest_categorical("feature_group", feature_groups)
    params = {
        **xgb_fixed,
        "max_depth": trial.suggest_int("max_depth", 1, 3),
        "n_estimators": trial.suggest_int("n_estimators", 50, 600, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.001, 100, log=True),
        "reg_alpha": trial.suggest_categorical("reg_alpha", [0, 0.01, 0.1, 1, 10]),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 30, log=True),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
    }
    predictions = dc.forecast_xgb(folds, group, params)
    scored = predictions.dropna(subset=["direction"])
    trial.set_user_attr("scoring_rows", len(scored))
    return float(scored["prediction"].eq(scored["direction"]).mean())


logistic_candidates = {
    f"C_{strength:g}": {"C": strength, "max_iter": 2000}
    for strength in (0.1, 1.0, 10.0)
}

## 6 - Choose features and tune the winner

First compare feature groups with equal tuning budgets, then inspect the results before continuing with the winning group. Every trial uses the same 72 monthly folds and pooled Data2 direction accuracy.

### Compare feature groups

Run 50 trials for each of the four groups (200 total), tuning numerical hyperparameters within the existing bounds. The table shows each group's best accuracy; the printed winner is the group selected for further tuning.

In [7]:
SCREENING_TRIALS_PER_GROUP = 50

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.NopPruner(),
)
for _ in range(SCREENING_TRIALS_PER_GROUP):
    for group in feature_groups:
        study.enqueue_trial({"feature_group": group})
screening_trials = SCREENING_TRIALS_PER_GROUP * len(feature_groups)
study.optimize(objective, n_trials=screening_trials)
screening_results = study.trials_dataframe()
screening_winner = min(study.trials, key=lambda trial: (-trial.value, trial.number))

display(
    screening_results.groupby("params_feature_group").agg(
        trials=("number", "size"), best_accuracy=("value", "max")
    )
)
print(
    f"Screening winner: {screening_winner.params['feature_group']}; "
    f"Data2 accuracy: {screening_winner.value:.2%}"
)

,trials,best_accuracy
params_feature_group,,
garch_pc_regimes,50,0.505648
market_garch,50,0.528239
market_macro_garch,50,0.536877
market_macro_garch_pc_regimes,50,0.530897


Screening winner: market_macro_garch; Data2 accuracy: 53.69%


### Tune the winning group

In [8]:
REFINEMENT_TRIALS = 350
REFINEMENT_GROUP = screening_winner.params["feature_group"]
N_TRIALS = screening_trials + REFINEMENT_TRIALS
for _ in range(REFINEMENT_TRIALS):
    study.enqueue_trial({"feature_group": REFINEMENT_GROUP})
study.optimize(objective, n_trials=REFINEMENT_TRIALS)

trial_results = study.trials_dataframe()
best_trial = min(
    (
        trial
        for trial in study.trials
        if trial.params["feature_group"] == REFINEMENT_GROUP
    ),
    key=lambda trial: (-trial.value, trial.number),
)
best_group = REFINEMENT_GROUP
best_params = {
    **xgb_fixed,
    **{
        name: value
        for name, value in best_trial.params.items()
        if name != "feature_group"
    },
}

display(
    trial_results.groupby("params_feature_group").agg(
        trials=("number", "size"), best_accuracy=("value", "max")
    )
)
display(pd.Series(best_trial.params, name="Selected value").to_frame())
print(f"Selected trial: {best_trial.number}; Data2 accuracy: {best_trial.value:.2%}")

,trials,best_accuracy
params_feature_group,,
garch_pc_regimes,50,0.505648
market_garch,50,0.528239
market_macro_garch,400,0.546179
market_macro_garch_pc_regimes,50,0.530897


,Selected value
feature_group,market_macro_garch
max_depth,1
n_estimators,100
learning_rate,0.154665
reg_lambda,0.255395
reg_alpha,1
min_child_weight,7.403685
subsample,0.96475
colsample_bytree,0.528581


Selected trial: 526; Data2 accuracy: 54.62%


### Evaluate and save the selected configuration

Refit the selected configuration across the monthly folds.

In [9]:
xgb_predictions = dc.forecast_xgb(folds, best_group, best_params)
scored = xgb_predictions.dropna(subset=["direction"])

display(
    pd.Series(
        {
            "Scoring dates": len(scored),
            "Actual up share": scored["direction"].mean(),
            "Predicted up share": scored["prediction"].mean(),
            "Direction accuracy": scored["prediction"].eq(scored["direction"]).mean(),
        }
    ).to_frame("Selected XGBoost")
)

direction_dir = data_dir / "direction"
direction_dir.mkdir(exist_ok=True)
xgb_predictions.to_parquet(direction_dir / "xgb_predictions_development.parquet")

,Selected XGBoost
Scoring dates,1505.000000
Actual up share,0.516944
Predicted up share,0.532226
Direction accuracy,0.546179


## 7 - Direction benchmarks

Use the selected XGBoost feature group and identical eligible rows. Always-up predicts 1. L2 logistic regression uses the same feature columns, with scaling fitted only on that month's training rows. Up means model output strictly above 0.5.

In [10]:
benchmark_predictions = dc.forecast_direction_benchmarks(
    folds, best_group, logistic_candidates
)
scoring_dates = xgb_predictions.index[xgb_predictions["direction"].notna()]
actual = xgb_predictions.loc[scoring_dates, "direction"].astype(int)

logistic_accuracy = (
    benchmark_predictions.loc[scoring_dates, list(logistic_candidates)]
    .eq(actual, axis=0)
    .mean()
)
best_logistic = logistic_accuracy.idxmax()
display(logistic_accuracy.to_frame("Data2 accuracy"))
print(f"Selected logistic setting: {best_logistic}")

,Data2 accuracy
C_0.1,0.510963
C_1,0.513621
C_10,0.518272


Selected logistic setting: C_10


In [11]:
directions = benchmark_predictions.loc[
    scoring_dates, ["always_up", best_logistic]
].rename(
    columns={
        "always_up": "Always-up",
        best_logistic: "L2 logistic",
    }
)
directions["XGBoost"] = xgb_predictions.loc[scoring_dates, "prediction"]
comparison = pd.DataFrame(
    {
        "Accuracy": directions.eq(actual, axis=0).mean(),
        "Predicted up share": directions.mean(),
    }
)
display(comparison)
print(f"Scoring dates: {len(actual)}; actual up share: {actual.mean():.2%}")

,Accuracy,Predicted up share
Always-up,0.516944,1.000000
L2 logistic,0.518272,0.613289
XGBoost,0.546179,0.532226


Scoring dates: 1505; actual up share: 51.69%


## 8 - Uncertainty of the accuracy differences

Adjacent five-row outcomes overlap, so daily correctness is dependent. Resample contiguous circular blocks of the paired correctness differences (XGBoost minus each benchmark); keep all models on the same sampled dates.

In [12]:
accuracy_uncertainty = dc.accuracy_difference_intervals(
    directions, actual, block_sizes=(5, 21, 63), reps=5000, seed=42
)

display(
    HTML(
        accuracy_uncertainty.to_html(
            formatters={
                "accuracy_gain_pp": "{:.2f}".format,
                "lower_95_pp": "{:.2f}".format,
                "upper_95_pp": "{:.2f}".format,
            }
        )
    )
)

XGBoost's estimated Data2 accuracy advantages are **+2.92 pp** over always-up and **+2.79 pp** over logistic regression. The estimates favor XGBoost, but all reported 95% intervals include zero, allowing no advantage or a disadvantage. These intervals hold the selected predictions fixed and do not account for repeated model selection on Data2.

## 9 - Phase 4 conclusion and frozen configuration

Retain XGBoost with market + macro + GARCH inputs (21 columns), without PCs or regimes. Its Data2 accuracy is **54.62%**, compared with **51.83%** for L2 logistic regression and **51.69%** for always-up. The estimated advantages favor XGBoost, but the paired block-bootstrap intervals include zero. Separately, selecting features and hyperparameters using Data2 can make the measured advantages optimistic; these intervals do not correct for that selection effect. Independent confirmation remains for Data3.

In [13]:
import json

frozen_config = {
    "feature_group": best_group,
    "feature_columns": list(
        next(iter(folds.values()))["inputs"]["train"][best_group].columns
    ),
    "xgb_params": best_params,
}
config_path = direction_dir / "selected_direction_config.json"
config_path.write_text(json.dumps(frozen_config, indent=2) + "\n", encoding="utf-8")
print(f"Saved {config_path.name}")

Saved selected_direction_config.json
